In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np
import os
from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1
from torch import Tensor
import random
from collections import defaultdict
from typing import Literal

In [12]:
image_name_to_person_id: dict[str, int] = {}
person_id_to_image_names: dict[int, list[str]] = defaultdict(list)

with open("identity_CelebA.txt", "r") as lines:
    for line in lines:
        image_name, person_id_str = line.split(" ")
        person_id = int(person_id_str.strip())
        image_name_to_person_id[image_name.strip()] = person_id
        person_id_to_image_names[person_id].append(image_name.strip())

person_ids_with_multiple_images: list[int] = [pid for pid, images in person_id_to_image_names.items() if len(images) > 1]
all_image_names = list(image_name_to_person_id.keys())

In [13]:
def get_balanced_training_data(
    total_pairs: int, 
    exclude_images: set[str] = set()
) -> list[tuple[str, str, int]]:
    """
    Generates a balanced dataset of image pairs.
    Aims for a 50/50 split between "same person" and "different people" pairs.
    """
    train_data = set()
    used_pairs = set() # To avoid (a,b) and (b,a) duplicates

    # --- 1. Generate "Same Person" pairs ---
    num_same_pairs = total_pairs // 2
    
    # Filter out persons whose images are all in the exclude set
    available_persons = [
        pid for pid in person_ids_with_multiple_images 
        if not all(img in exclude_images for img in person_id_to_image_names[pid])
    ]

    while len(train_data) < num_same_pairs:
        # Pick a random person who has at least two images
        person_id = random.choice(available_persons)
        
        # Get images for this person, excluding those in the validation set
        possible_images = [img for img in person_id_to_image_names[person_id] if img not in exclude_images]
        
        if len(possible_images) < 2:
            continue

        # Sample two different images for this person
        img1, img2 = random.sample(possible_images, 2)
        
        # Add to sets to prevent duplicates
        pair = tuple(sorted((img1, img2)))
        if pair not in used_pairs:
            train_data.add((img1, img2, 1)) # Label 1 for "same"
            used_pairs.add(pair)
            
    # --- 2. Generate "Different People" pairs ---
    available_images = [img for img in all_image_names if img not in exclude_images]

    while len(train_data) < total_pairs:
        # Pick two random images
        img1, img2 = random.sample(available_images, 2)

        # Ensure they are from different people
        if image_name_to_person_id[img1] != image_name_to_person_id[img2]:
            pair = tuple(sorted((img1, img2)))
            if pair not in used_pairs:
                train_data.add((img1, img2, 0)) # Label 0 for "different"
                used_pairs.add(pair)

    # Shuffle the final list and return
    final_data = list(train_data)
    random.shuffle(final_data)
    return final_data

In [15]:
DATA_DIR = "img_align_celeba/"

result_type = Literal["Same person"] | Literal["Different people"]

class FaceVerificationMLP(nn.Module):
    '''Define the MLP model that takes the difference between two embeddings'''
    def __init__(self, input_dim=512):
        super(FaceVerificationMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Added dropout for regularization
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.5), # Added dropout for regularization
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.model(x)
   

# Optional Preprocessing: Resize, smooth, convert to tensor, and normalize the image if they require it in one of the tasks. Note: this is a sample transoformation, modify it accoringly to the task
default_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
]) 

# Load MTCNN for face detection, note: you should adjust the size wrt data and model, it can be done directly here or in the transform defined above
mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device='cuda' if torch.cuda.is_available() else 'cpu')

# Load FaceNet for embedding extraction (we'll be using a pretrained model)
facenet: InceptionResnetV1 = InceptionResnetV1(pretrained='vggface2', device='cuda' if torch.cuda.is_available() else 'cpu').eval()
    
def get_embedding(image_path):
    '''Function to get face embedding from an image path'''
    try:
        img = Image.open(DATA_DIR + image_path).convert('RGB')
    except FileNotFoundError:
        print(f"Error: Could not find image {image_path}")
        return None

    # mtcnn() returns a tensor on the device it was initialized with (e.g., 'cuda')
    face = mtcnn(img) 

    if face is None:
        # Fallback for MTCNN failure: resize and center crop
        # print(f"Warning: No face detected in {image_path}, using fallback.")
        transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(), # This creates a tensor on the CPU
        ])
        face = transform(img)
    # At this point, `face` is a tensor (either from mtcnn or from fallback)
    if isinstance(face, torch.Tensor):
        face = face.to(facenet.device)
    else:
        raise RuntimeError("Face could not be converted to tensor.")
    
    # The input tensor `face` is already on the correct device, so this call is fine.
    face_embedding = facenet(face.unsqueeze(0)) 
    return face_embedding.detach()


def get_diff_vector(img1_path, img2_path) -> Tensor | None:
    '''Function to compare two images and get the absolute difference vector'''
    emb1 = get_embedding(img1_path)
    emb2 = get_embedding(img2_path)
    if emb1 is None or emb2 is None:
        return None
    return torch.abs(emb1 - emb2)


def predict_same_person(img1_path, img2_path, model) -> result_type:
    '''Prediction Example, for additional experiments you may want to return the decision in numeric form or add model certenity'''
    model.eval()
    with torch.no_grad():
        diff = get_diff_vector(img1_path, img2_path)
        if diff is None:
            return 'Different people' # Default prediction on error
        diff = diff.to(next(model.parameters()).device)
        output = model(diff)
        _, predicted = torch.max(output, 1)
    return 'Same person' if predicted.item() == 1 else 'Different people'


def augment_image(image, augment_type="gaussian_noise"):
    '''Example augmentation function'''
    if augment_type == "gaussian_noise":
        image_np = np.array(image).astype(np.float32)
        noise = np.random.normal(0, 25, image_np.shape)
        noisy_image = image_np + noise
        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
        augmented = Image.fromarray(noisy_image)
    elif augment_type == "blur":
        augmented = image.filter(ImageFilter.GaussianBlur(radius=3))
    elif augment_type == "increased_lighting":
        enhancer = ImageEnhance.Brightness(image)
        augmented = enhancer.enhance(1.5)
    else:
        augmented = image
    return augmented

In [ ]:
def train_model(train_data: list[tuple[str, str, int]]) -> FaceVerificationMLP:
    # Prepare training tensors
    X_train = []
    y_train = []
    
    print("Generating embeddings for training data...")
    for img1, img2, label in train_data:
        diff = get_diff_vector(img1, img2)
        if diff is not None:
            X_train.append(diff.squeeze(0))
            y_train.append(label)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    X_train = torch.stack(X_train).to(device)
    y_train = torch.tensor(y_train).to(device)

    # Define model, loss, optimizer
    model = FaceVerificationMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Simple batching for training
    batch_size = 64
    epochs = 20
    
    print("Starting model training...")
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(X_train.size()[0])
        epoch_loss = 0.0
        
        for i in range(0, X_train.size()[0], batch_size):
            optimizer.zero_grad()
            indices = permutation[i:i+batch_size]
            batch_X, batch_y = X_train[indices], y_train[indices]

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        # print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / (len(X_train)/batch_size):.4f}")
        
    return model

In [ ]:
VALIDATION_DATA_COUNT = 200

ns = [10, 100, 500, 1000, 5000]

for n in ns:
    print("-" * 50)
    print(f"Starting experiment with N = {n}")
    
    print("Generating training data...")
    train_data: list[tuple[str, str, int]] = get_balanced_training_data(n) 
    
    exclude_images = set(img for pair in train_data for img in pair[:2])
    
    print("Generating validation data...")
    validation_data = get_balanced_training_data(VALIDATION_DATA_COUNT, exclude_images=exclude_images) 
    
    model = train_model(train_data)
        
    print("Evaluating model...")
    correct_count = 0
    total_valid = 0
    
    for img1, img2, label in validation_data:
        result = predict_same_person(img1, img2, model)
        is_correct = (result == 'Same person' and label == 1) or \
                        (result == 'Different people' and label == 0)
        if is_correct:
            correct_count += 1
        total_valid += 1
    
    accuracy = (correct_count / total_valid) * 100 if total_valid > 0 else 0
    print("\n--- Results for N =", n, "---")
    print(f"Validation Accuracy: {accuracy:.2f}% ({correct_count}/{total_valid})")
    print("-" * 50 + "\n")

--------------------------------------------------
Starting experiment with N = 10
Generating training data...
Generating validation data...
Generating embeddings for training data...
Starting model training...
Evaluating model...

--- Results for N = 10 ---
Validation Accuracy: 50.00% (100/200)
--------------------------------------------------

--------------------------------------------------
Starting experiment with N = 100
Generating training data...
Generating validation data...
Generating embeddings for training data...
  Processed 100/100 pairs...
Starting model training...
Evaluating model...

--- Results for N = 100 ---
Validation Accuracy: 70.50% (141/200)
--------------------------------------------------

--------------------------------------------------
Starting experiment with N = 500
Generating training data...
Generating validation data...
Generating embeddings for training data...
  Processed 100/500 pairs...
  Processed 200/500 pairs...
  Processed 300/500 pairs...